# Tokenizing with tiktoken and Understanding the Illusion of Memory

This notebook explores how LLMs break test into tokens and why chatbots only appear to have memory when previous messages are sent again in the request.

In [ ]:
%pip install tiktoken openai python-dotenv

In [2]:
import os
import tiktoken
from dotenv import load_dotenv
from openai import OpenAI

In [3]:
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY is not set")

client = OpenAI(api_key=api_key)

## What is a token?

A token is a unit of text used by an LLM. It may be:
- a full word
- a part of a word
- punctuation
- whitespace

LLMs do not read text as full sentences. They read token sequences. 

In [4]:
encoding = tiktoken.encoding_for_model("gpt-4o-mini")

text = "AI engineering is powerful."
tokens = encoding.encode(text)

print("Text:", text)
print("Token IDs:", tokens)
print("Number of tokens:", len(tokens))

Text: AI engineering is powerful.
Token IDs: [17527, 16411, 382, 11629, 13]
Number of tokens: 5


In [ ]:
# decode tokens back to text
decode_text = encoding.decode(tokens)
print(decode_text)

AI engineering is powerful.


In [6]:
# inspect each token
for token_id in tokens:
    token_text = encoding.decode([token_id])
    print(token_id, repr(token_text))

17527 'AI'
16411 ' engineering'
382 ' is'
11629 ' powerful'
13 '.'


In [7]:
# Compare toekn counts
examples = [
    "Hello world",
    "Artificial intelligence is changing software engineering.",
    "Prior authorizationrequires approval before certain medications are covered.",
    "😊🚀🔥"
]

for example in examples:
    token_ids = encoding.encode(example)
    print("=" * 80)
    print("Text:", example)
    print("Tokens:", token_ids)
    print("Token count:", len(token_ids))

Text: Hello world
Tokens: [13225, 2375]
Token count: 2
Text: Artificial intelligence is changing software engineering.
Tokens: [186671, 22990, 382, 13784, 4305, 16411, 13]
Token count: 7
Text: Prior authorizationrequires approval before certain medications are covered.
Tokens: [59235, 42609, 93000, 21981, 2254, 4454, 34576, 553, 13083, 13]
Token count: 10
Text: 😊🚀🔥
Tokens: [102630, 112927, 222, 96606]
Token count: 4


## Why Token Count Matters

Token count affects:

- cost
- latency
- context window usage
- how much conversation histriy can fit into a request

Longer prompts cost more and leave less room for the model's answer.

In [10]:
# Chat completion without memory
response = client.chat.completions.create(
    model = "gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "My name is Yugeen."}
    ],
    temperature=0.3,
)

print(response.choices[0].message.content)

Hello, Yugeen! How can I assist you today?


In [11]:
# now ask separately
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is my name?"}
    ],
    temperature=0.3,
)

print(response.choices[0].message.content)

I'm sorry, but I don't have access to personal information about you unless you share it with me. How can I assist you today?


## The Illusion of Memory

The model does not automatically remember previous API calls.

Each API request is stateless.

A chatbot appears to have memory only because the application sends previous messages again in the `messages` list.

In [13]:
from openai.types.chat import ChatCompletionMessageParam

# Chat completion with memory
messages: list[ChatCompletionMessageParam] = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "My name is Yugeen."},
    {"role": "assistant", "content": "Nice to meet you, Yugeen."},
    {"role": "user", "content": "What is my name?"},
]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    temperature=0.3,
)

print(response.choices[0].message.content)

Your name is Yugeen.


In [15]:
## Count tokens in chat history
conversation_text = ""

for message in messages:
    role = str(message.get("role", "unknown"))
    content = message.get("content")

    if isinstance(content, str):
        content_text = content
    elif content is None:
        content_text = ""
    else:
        # Handle structured content parts by joining available text fields.
        content_text = " ".join(
            part.get("text", "")
            for part in content
            if isinstance(part, dict) and isinstance(part.get("text"), str)
        )

    conversation_text += f"{role}: {content_text}\n"

tokens = encoding.encode(conversation_text)

print(conversation_text)
print("Approx token count:", len(tokens))

system: You are a helpful assistant.
user: My name is Yugeen.
assistant: Nice to meet you, Yugeen.
user: What is my name?

Approx token count: 35


In [16]:
# show memory gets expensive
messages = [
    {"role": "system", "content": "You are a helpful assistant."}
]

for i in range(1, 11):
    messages.append({"role": "user", "content": f"This is message number {i}. Remember it."})
    messages.append({"role": "assistant", "content": f"Got it. I will remember message number {i}."})

conversation_text = "\n".join(
    f"{m['role']}: {m['content']}" for m in messages
)

token_count = len(encoding.encode(conversation_text))

print("Messages:", len(messages))
print("Approx token count:", token_count)

Messages: 21
Approx token count: 258


## Key Takeaways

- LLMs process text as tokens, not full words.
- Tokens affect cost, latency, and context limits.
- API calls are stateless by default.
- Chatbots only appear to remember because previous messages are resent.
- More memory means more tokens.
- Real AI systems must manage conversation history carefully.